In [1]:
# Install JAX with GPU
!pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Run full study (all L and beta values)
!python colab_rg_blocking_study.py

# OR run single test configuration first
# Edit line 531 to: result = run_rg_blocking_study(L=8, beta=5.0, n_sweeps=500, block_levels=2)

Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 18.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82
python3: can't open file '/content/colab_rg_blocking_study.py': [Errno 2] No such file or directory


In [3]:
# --- COLAB SETUP ---
# !pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
# -------------------

"""
RG BLOCKING TRANSFORMATION STUDY
=================================
Based on GPT-5.1 consultation guidance (next_simulation_guidance.json)

This script implements Balaban's renormalization group blocking transformation
and measures how defect density and convexity properties evolve under RG flow.

Key Question: Does defect density decrease exponentially under blocking while
the convexity radius remains ≥ r_crit = 1.9248?

This directly validates the core assumption of Balaban's mass gap proof:
that the polymer activity ̘ remains small at ALL RG scales.
"""

import jax
import jax.numpy as jnp
from jax import random, jit, vmap
import numpy as np
import time
import json
from datetime import datetime

# --- CONFIGURATION ---
jax.config.update("jax_enable_x64", False)  # Use float32/complex64 for speed
print(f"JAX Device: {jax.devices()[0]}")
print(f"JAX Backend: {jax.default_backend()}")

# --- SU(2) ALGEBRA ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-12)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

@jit
def log_map_su2(U):
    """Map SU(2) element U to R^3 vector (inverse of exp_map)."""
    tr = jnp.trace(U)
    tr_clamped = jnp.clip(jnp.real(tr), -2.0, 2.0)
    theta = jnp.arccos(0.5 * tr_clamped)

    # Handle small angles
    safe_theta = jnp.where(theta < 1e-8, 1e-8, theta)
    coeff = theta / (2.0 * jnp.sin(safe_theta))

    # Extract Pauli components
    alpha_x = -1j * coeff * (U[0, 1] - U[1, 0]) / 2.0
    alpha_y = -1j * coeff * (U[0, 1] + U[1, 0]) / (2.0 * 1j)
    alpha_z = -1j * coeff * (U[0, 0] - U[1, 1]) / 2.0

    return jnp.array([jnp.real(alpha_x), jnp.real(alpha_y), jnp.real(alpha_z)])

# --- LATTICE SETUP ---
def init_lattice_cold(L):
    """Initialize all links to identity (cold start)"""
    Dim = 4
    NumLinks = L**Dim * Dim
    shape = (L, L, L, L, Dim, 2, 2)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape(shape)

def make_checkerboard_mask(L):
    """Create checkerboard pattern for parallel updates"""
    x = jnp.arange(L)
    X, Y, Z, T = jnp.meshgrid(x, x, x, x, indexing='ij')
    parity = (X + Y + Z + T) % 2
    return parity

# --- PARALLEL STAPLES ---
@jit
def get_staples_parallel(U):
    """Compute staples for all links in parallel"""
    staples = jnp.zeros_like(U)
    for nu in range(4):
        for mu in range(4):
            if mu == nu: continue
            U_xpmu_nu = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U_xpnu_mu = jnp.roll(U[..., mu, :, :], -1, axis=nu)
            U_x_nu = U[..., nu, :, :]
            term1 = U_xpmu_nu @ jnp.conjugate(jnp.swapaxes(U_xpnu_mu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_x_nu, -1, -2))

            U_xmnu_nu = jnp.roll(U[..., nu, :, :], 1, axis=nu)
            U_xmnu_mu = jnp.roll(U[..., mu, :, :], 1, axis=nu)
            U_xpmu_mnu_nu = jnp.roll(jnp.roll(U[..., nu, :, :], -1, axis=mu), 1, axis=nu)
            term2 = jnp.conjugate(jnp.swapaxes(U_xpmu_mnu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_xmnu_mu, -1, -2)) @ U_xmnu_nu

            staples = staples.at[..., mu, :, :].add(term1 + term2)
    return staples

# --- CHECKERBOARD UPDATE ---
@jit
def update_checkerboard(key, U, beta, parity_mask):
    """Metropolis update on one checkerboard sublattice"""
    staples = get_staples_parallel(U)

    key, subkey = random.split(key)
    alpha = random.normal(subkey, U.shape[:-2] + (3,)) * 0.1
    alpha_flat = alpha.reshape((-1, 3))
    dU_flat = vmap(exp_map_pauli)(alpha_flat)
    dU = dU_flat.reshape(U.shape)
    U_new = dU @ U

    tr_old = jnp.trace(U @ staples, axis1=-2, axis2=-1)
    tr_new = jnp.trace(U_new @ staples, axis1=-2, axis2=-1)
    S_old = -0.5 * beta * jnp.real(tr_old)
    S_new = -0.5 * beta * jnp.real(tr_new)
    dS = S_new - S_old

    key, subkey = random.split(key)
    rand_vals = random.uniform(subkey, dS.shape)
    accept = rand_vals < jnp.exp(-dS)

    mask_broad = jnp.expand_dims(parity_mask, axis=-1)
    mask_broad = jnp.tile(mask_broad, (1, 1, 1, 1, 4))
    do_update = jnp.logical_and(accept, mask_broad == 1)
    do_update_mat = jnp.expand_dims(do_update, axis=(-1, -2))

    U_final = jnp.where(do_update_mat, U_new, U)

    n_active = jnp.sum(mask_broad)
    n_accepted = jnp.sum(jnp.where(mask_broad==1, accept, 0.0))
    return key, U_final, n_accepted / (n_active + 1e-10)

# --- RG BLOCKING TRANSFORMATION ---

def block_link_scale2_impl(U, x, y, z, t, mu):
    """
    Compute blocked link variable at (x,y,z,t) in direction mu
    using scale-2 blocking (average over 2^(d-1) paths in the hypercube).

    For 4D lattice, we average over 8 parallel transport paths connecting
    the two block points separated by direction mu.
    """
    # Fine lattice coordinates
    x0, y0, z0, t0 = 2*x, 2*y, 2*z, 2*t

    # The blocked link connects block (x,y,z,t) to block (x+d_mu, y+d_vy, z+d_z, t+d_t)
    # We average over paths through the 2x2x2x2 hypercube

    # Simple blocking: average of 2 parallel links in direction mu
    # More sophisticated: average over staple-based paths

    if mu == 0:  # x-direction
        # Average U(x0,y0,z0,t0,0) and U(x0+1,y0,z0,t0,0) and paths through other dirs
        U1 = U[x0, y0, z0, t0, 0]
        U2 = U[x0+1, y0, z0, t0, 0]
        # Simple average for now (can improve with staples)
        U_blocked = 0.5 * (U1 + U2)
    elif mu == 1:  # y-direction
        U1 = U[x0, y0, z0, t0, 1]
        U2 = U[x0, y0+1, z0, t0, 1]
        U_blocked = 0.5 * (U1 + U2)
    elif mu == 2:  # z-direction
        U1 = U[x0, y0, z0, t0, 2]
        U2 = U[x0, y0, z0+1, t0, 2]
        U_blocked = 0.5 * (U1 + U2)
    else:  # t-direction
        U1 = U[x0, y0, z0, t0, 3]
        U2 = U[x0, y0, z0, t0+1, 3]
        U_blocked = 0.5 * (U1 + U2)

    # Project back to SU(2) using log-exp trick
    # Convert to algebra, rescale, convert back
    alpha = log_map_su2(U_blocked)
    return exp_map_pauli(alpha)

block_link_scale2 = jit(block_link_scale2_impl, static_argnums=5)

def perform_rg_blocking(U):
    """
    Perform scale-2 RG blocking transformation.
    Input: U with shape (L, L, L, L, 4, 2, 2)
    Output: U_blocked with shape (L//2, L//2, L//2, L//2, 4, 2, 2)
    """
    L = U.shape[0]
    if L % 2 != 0:
        raise ValueError("Lattice size must be even for scale-2 blocking")

    L_blocked = L // 2
    U_blocked = jnp.zeros((L_blocked, L_blocked, L_blocked, L_blocked, 4, 2, 2), dtype=jnp.complex64)

    # Loop over blocked lattice sites
    for x in range(L_blocked):
        for y in range(L_blocked):
            for z in range(L_blocked):
                for t in range(L_blocked):
                    for mu in range(4):
                        U_blocked = U_blocked.at[x, y, z, t, mu].set(
                            block_link_scale2(U, x, y, z, t, mu)
                        )

    return U_blocked

# --- OBSERVABLES ---

@jit
def measure_plaquette_defects(U, r_crit):
    """Measure fraction of plaquettes violating convexity bound"""
    Vol = U.shape[0]**4
    defects = 0.0
    total_plaqs = 0.0

    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))

            tr_vals = jnp.real(jnp.trace(P, axis1=-2, axis2=-1))
            tr_vals = jnp.clip(tr_vals, -2.0, 2.0)
            thetas = jnp.arccos(0.5 * tr_vals)

            defects += jnp.sum(thetas > r_crit)
            total_plaqs += Vol

    return defects / total_plaqs

@jit
def measure_plaquette_action_statistics(U, beta):
    """Measure mean and std of plaquette action distribution"""
    Vol = U.shape[0]**4
    actions = []

    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
            tr_vals = jnp.real(jnp.trace(P, axis1=-2, axis2=-1))

            # S_plaq = -beta/2 * Re Tr P
            S_plaq = -0.5 * beta * tr_vals
            actions.append(S_plaq.flatten())

    all_actions = jnp.concatenate(actions)
    return jnp.mean(all_actions), jnp.std(all_actions)

@jit
def estimate_local_convexity_radius(U, beta, n_samples=100):
    """
    Estimate effective local convexity radius by sampling small perturbations
    around typical configurations and checking Hessian eigenvalues.

    Returns: estimate of the radius where action is locally convex
    """
    # Sample random links
    L = U.shape[0]
    key = random.PRNGKey(42)

    # Take a few random links and perturb them slightly
    # Compute local curvature (second derivative of action)
    # This is a simplified proxy - full implementation would compute Hessian

    # For now, return plaquette-based estimate
    # Effective radius ~ 1/sqrt(local curvature)

    # Compute typical plaquette trace
    avg_plaq_trace = 0.0
    count = 0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
            avg_plaq_trace += jnp.mean(jnp.real(jnp.trace(P, axis1=-2, axis2=-1)))
            count += 1

    avg_plaq_trace /= count

    # Estimate curvature: higher plaq trace -> tighter binding -> larger effective radius
    # r_eff ~ arccos(trace/2) measures typical "distance from identity"
    r_eff = jnp.arccos(jnp.clip(0.5 * avg_plaq_trace, -1.0, 1.0))

    return r_eff

# --- MAIN RG STUDY DRIVER ---

def run_rg_blocking_study(L, beta, n_sweeps=2000, block_levels=2):
    """
    Run RG blocking study for a single (L, beta) point.

    Args:
        L: Lattice size (must be divisible by 2^block_levels)
        beta: Coupling parameter
        n_sweeps: Number of measurement sweeps
        block_levels: Number of RG blocking levels to perform
    """
    print("\n" + "="*70)
    print(f"RG BLOCKING STUDY: L={L}^4, Beta={beta:.4f}")
    print("="*70)

    r_crit = 1.9248  # Balaban's convexity threshold

    # Check lattice size
    if L % (2**block_levels) != 0:
        raise ValueError(f"L={L} must be divisible by 2^{block_levels}={2**block_levels}")

    # Initialize
    key = random.PRNGKey(42)
    U = init_lattice_cold(L)
    parity_mask = make_checkerboard_mask(L)

    @jit
    def full_sweep(key, U):
        key, U, acc1 = update_checkerboard(key, U, beta, parity_mask)
        key, U, acc2 = update_checkerboard(key, U, beta, 1 - parity_mask)
        return key, U, (acc1 + acc2) / 2.0

    # Thermalization
    print("\nThermalizing...")
    therm_sweeps = min(500, n_sweeps // 2)
    t0 = time.time()
    key, U, acc = full_sweep(key, U)
    acc.block_until_ready()
    print(f"JIT compile: {time.time()-t0:.2f}s")

    for i in range(therm_sweeps):
        key, U, acc = full_sweep(key, U)
        if i % 100 == 0:
            acc.block_until_ready()
            print(f"  Thermalization sweep {i}/{therm_sweeps}, acc={float(acc):.2f}")

    print(f"Thermalization complete: {time.time()-t0:.2f}s")

    # Measurements
    print(f"\nMeasuring unblocked lattice ({n_sweeps} sweeps)...")
    t0 = time.time()

    measurements_unblocked = []

    for i in range(n_sweeps):
        key, U, acc = full_sweep(key, U)

        if i % 50 == 0:
            # Measure observables
            rho_defect = measure_plaquette_defects(U, r_crit)
            r_eff = estimate_local_convexity_radius(U, beta)
            S_mean, S_std = measure_plaquette_action_statistics(U, beta)

            measurements_unblocked.append({
                "sweep": i,
                "defect_density": float(rho_defect),
                "convexity_radius": float(r_eff),
                "plaq_action_mean": float(S_mean),
                "plaq_action_std": float(S_std),
                "acceptance": float(acc)
            })

            if i % 200 == 0 and i > 0:
                print(f"  Sweep {i}: defect={float(rho_defect):.6f}, r_eff={float(r_eff):.4f}, acc={float(acc):.2f}")

    print(f"Unblocked measurements: {time.time()-t0:.2f}s")

    # Perform RG blocking and measure at each level
    blocked_results = []
    U_current = U

    for level in range(1, block_levels + 1):
        print(f"\n--- RG Blocking Level {level} ---")
        print(f"Blocking from L={U_current.shape[0]} to L={U_current.shape[0]//2}...")

        t0 = time.time()
        U_blocked = perform_rg_blocking(U_current)
        print(f"Blocking transformation: {time.time()-t0:.2f}s")

        # Measure blocked lattice
        print("Measuring blocked lattice...")
        rho_defect_blocked = measure_plaquette_defects(U_blocked, r_crit)
        r_eff_blocked = estimate_local_convexity_radius(U_blocked, beta)
        S_mean_blocked, S_std_blocked = measure_plaquette_action_statistics(U_blocked, beta)

        result = {
            "level": level,
            "L_blocked": U_blocked.shape[0],
            "defect_density": float(rho_defect_blocked),
            "convexity_radius": float(r_eff_blocked),
            "plaq_action_mean": float(S_mean_blocked),
            "plaq_action_std": float(S_std_blocked)
        }

        blocked_results.append(result)

        print(f"  L={result['L_blocked']}: defect={result['defect_density']:.6f}, r_eff={result['convexity_radius']:.4f}")

        U_current = U_blocked

    # Compute statistics
    skip = len(measurements_unblocked) // 10
    analysis_data = measurements_unblocked[skip:]

    summary = {
        "L": L,
        "beta": beta,
        "n_sweeps": n_sweeps,
        "block_levels": block_levels,
        "r_crit": r_crit,
        "unblocked": {
            "mean_defect_density": float(np.mean([m["defect_density"] for m in analysis_data])),
            "std_defect_density": float(np.std([m["defect_density"] for m in analysis_data])),
            "mean_convexity_radius": float(np.mean([m["convexity_radius"] for m in analysis_data])),
            "mean_plaq_action": float(np.mean([m["plaq_action_mean"] for m in analysis_data])),
            "std_plaq_action": float(np.mean([m["plaq_action_std"] for m in analysis_data])),
        },
        "blocked_levels": blocked_results,
        "measurements_unblocked": measurements_unblocked
    }

    # Estimate ̘_eff
    print("\n" + "="*70)
    print("EFFECTIVE POLYMER ACTIVITY ESTIMATE")
    print("="*70)

    for i, result in enumerate(blocked_results):
        level = result["level"]
        rho = result["defect_density"]

        # ̘_eff ≤ ̘_defect × exp(-ΔS_local)
        # Typical defect action excess ~ beta (rough estimate)
        Delta_S = beta * 0.5  # Conservative estimate
        kappa_eff = rho * np.exp(-Delta_S)

        print(f"Level {level}: ̘={rho:.2e}, ̘_eff ≤ {kappa_eff:.2e}")
        result["kappa_eff_estimate"] = float(kappa_eff)

    print("\nBalaban's proof requires ̘ to remain small at ALL scales.")
    print("Validation: ̘_eff should DECREASE under blocking.")
    print("="*70)

    return summary

# --- MULTI-PARAMETER SCAN ---

def run_full_rg_study():
    """
    Run RG blocking study for multiple parameters as recommended by GPT-5.1.

    Parameters from guidance:
    - L: [8, 12, 16]
    - Beta: 3 values from 4.6 to 5.4
    - n_sweeps: 2000
    """

    print("="*70)
    print("COMPLETE RG BLOCKING STUDY")
    print("Based on GPT-5.1 Consultation Guidance")
    print("="*70)

    # Parameters
    lattice_sizes = [8, 12, 16]
    beta_values = [4.6, 5.0, 5.4]
    n_sweeps = 2000

    all_results = []

    for L in lattice_sizes:
        # Determine max blocking levels
        max_levels = int(np.log2(L)) - 1  # Keep at least L=4 after blocking
        max_levels = min(max_levels, 2)  # Limit to 2 levels for reasonable compute

        for beta in beta_values:
            print(f"\n{'#'*70}")
            print(f"# CONFIGURATION: L={L}, Beta={beta:.2f}")
            print(f"{'#'*70}")

            result = run_rg_blocking_study(
                L=L,
                beta=beta,
                n_sweeps=n_sweeps,
                block_levels=max_levels
            )

            all_results.append(result)

    # Save complete results
    output = {
        "study_type": "rg_blocking_multi_parameter",
        "timestamp": datetime.now().isoformat(),
        "gpt51_guidance": "next_simulation_guidance.json",
        "results": all_results
    }

    print("\n" + "="*70)
    print("COMPLETE RESULTS JSON")
    print("="*70)
    print(json.dumps(output, indent=2))

    return output

# --- MAIN EXECUTION ---

if __name__ == "__main__":
    # Option 1: Run single configuration for testing
    # result = run_rg_blocking_study(L=8, beta=5.0, n_sweeps=500, block_levels=2)

    # Option 2: Run full multi-parameter study (recommended)
    results = run_full_rg_study()

    print("\n✅ RG Blocking Study Complete!")
    print("Copy the JSON output to analyze ̘_eff evolution under blocking.")

JAX Device: cuda:0
JAX Backend: gpu
COMPLETE RG BLOCKING STUDY
Based on GPT-5.1 Consultation Guidance

######################################################################
# CONFIGURATION: L=8, Beta=4.60
######################################################################

RG BLOCKING STUDY: L=8^4, Beta=4.6000

Thermalizing...
JIT compile: 4.76s
  Thermalization sweep 0/500, acc=0.70
  Thermalization sweep 100/500, acc=0.70
  Thermalization sweep 200/500, acc=0.70
  Thermalization sweep 300/500, acc=0.71
  Thermalization sweep 400/500, acc=0.70
Thermalization complete: 8.67s

Measuring unblocked lattice (2000 sweeps)...
  Sweep 200: defect=0.000041, r_eff=0.5978, acc=0.71
  Sweep 400: defect=0.000000, r_eff=0.5961, acc=0.70
  Sweep 600: defect=0.000000, r_eff=0.5954, acc=0.70
  Sweep 800: defect=0.000000, r_eff=0.6018, acc=0.71
  Sweep 1000: defect=0.000041, r_eff=0.5969, acc=0.70
  Sweep 1200: defect=0.000000, r_eff=0.5979, acc=0.71
  Sweep 1400: defect=0.000000, r_eff=0.5970, acc

KeyboardInterrupt: 

In [4]:
# --- COLAB SETUP ---
# !pip install jax[cuda12] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
# -------------------

"""
RG BLOCKING TRANSFORMATION STUDY
=================================
Based on GPT-5.1 consultation guidance (next_simulation_guidance.json)

This script implements Balaban's renormalization group blocking transformation
and measures how defect density and convexity properties evolve under RG flow.

Key Question: Does defect density decrease exponentially under blocking while
the convexity radius remains ≥ r_crit = 1.9248?

This directly validates the core assumption of Balaban's mass gap proof:
that the polymer activity κ remains small at ALL RG scales.
"""

import jax
import jax.numpy as jnp
from jax import random, jit, vmap
import numpy as np
import time
import json
from datetime import datetime

# --- CONFIGURATION ---
jax.config.update("jax_enable_x64", False)  # Use float32/complex64 for speed
print(f"JAX Device: {jax.devices()[0]}")
print(f"JAX Backend: {jax.default_backend()}")

# --- SU(2) ALGEBRA ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-12)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

@jit
def log_map_su2(U):
    """Map SU(2) element U to R^3 vector (inverse of exp_map)."""
    tr = jnp.trace(U)
    tr_clamped = jnp.clip(jnp.real(tr), -2.0, 2.0)
    theta = jnp.arccos(0.5 * tr_clamped)

    # Handle small angles
    safe_theta = jnp.where(theta < 1e-8, 1e-8, theta)
    coeff = theta / (2.0 * jnp.sin(safe_theta))

    # Extract Pauli components
    alpha_x = -1j * coeff * (U[0, 1] - U[1, 0]) / 2.0
    alpha_y = -1j * coeff * (U[0, 1] + U[1, 0]) / (2.0 * 1j)
    alpha_z = -1j * coeff * (U[0, 0] - U[1, 1]) / 2.0

    return jnp.array([jnp.real(alpha_x), jnp.real(alpha_y), jnp.real(alpha_z)])

# --- LATTICE SETUP ---
def init_lattice_cold(L):
    """Initialize all links to identity (cold start)"""
    Dim = 4
    NumLinks = L**Dim * Dim
    shape = (L, L, L, L, Dim, 2, 2)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape(shape)

def make_checkerboard_mask(L):
    """Create checkerboard pattern for parallel updates"""
    x = jnp.arange(L)
    X, Y, Z, T = jnp.meshgrid(x, x, x, x, indexing='ij')
    parity = (X + Y + Z + T) % 2
    return parity

# --- PARALLEL STAPLES ---
@jit
def get_staples_parallel(U):
    """Compute staples for all links in parallel"""
    staples = jnp.zeros_like(U)
    for nu in range(4):
        for mu in range(4):
            if mu == nu: continue
            U_xpmu_nu = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U_xpnu_mu = jnp.roll(U[..., mu, :, :], -1, axis=nu)
            U_x_nu = U[..., nu, :, :]
            term1 = U_xpmu_nu @ jnp.conjugate(jnp.swapaxes(U_xpnu_mu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_x_nu, -1, -2))

            U_xmnu_nu = jnp.roll(U[..., nu, :, :], 1, axis=nu)
            U_xmnu_mu = jnp.roll(U[..., mu, :, :], 1, axis=nu)
            U_xpmu_mnu_nu = jnp.roll(jnp.roll(U[..., nu, :, :], -1, axis=mu), 1, axis=nu)
            term2 = jnp.conjugate(jnp.swapaxes(U_xpmu_mnu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_xmnu_mu, -1, -2)) @ U_xmnu_nu

            staples = staples.at[..., mu, :, :].add(term1 + term2)
    return staples

# --- CHECKERBOARD UPDATE ---
@jit
def update_checkerboard(key, U, beta, parity_mask):
    """Metropolis update on one checkerboard sublattice"""
    staples = get_staples_parallel(U)

    key, subkey = random.split(key)
    alpha = random.normal(subkey, U.shape[:-2] + (3,)) * 0.1
    alpha_flat = alpha.reshape((-1, 3))
    dU_flat = vmap(exp_map_pauli)(alpha_flat)
    dU = dU_flat.reshape(U.shape)
    U_new = dU @ U

    tr_old = jnp.trace(U @ staples, axis1=-2, axis2=-1)
    tr_new = jnp.trace(U_new @ staples, axis1=-2, axis2=-1)
    S_old = -0.5 * beta * jnp.real(tr_old)
    S_new = -0.5 * beta * jnp.real(tr_new)
    dS = S_new - S_old

    key, subkey = random.split(key)
    rand_vals = random.uniform(subkey, dS.shape)
    accept = rand_vals < jnp.exp(-dS)

    mask_broad = jnp.expand_dims(parity_mask, axis=-1)
    mask_broad = jnp.tile(mask_broad, (1, 1, 1, 1, 4))
    do_update = jnp.logical_and(accept, mask_broad == 1)
    do_update_mat = jnp.expand_dims(do_update, axis=(-1, -2))

    U_final = jnp.where(do_update_mat, U_new, U)

    n_active = jnp.sum(mask_broad)
    n_accepted = jnp.sum(jnp.where(mask_broad==1, accept, 0.0))
    return key, U_final, n_accepted / (n_active + 1e-10)

# --- RG BLOCKING TRANSFORMATION ---

@jit
def perform_rg_blocking(U):
    """
    Perform scale-2 RG blocking transformation using vectorized operations.

    Logic:
    The blocked link U_blocked(x_coarse, mu) corresponds to the product of two
    fine links connecting 2*x_coarse to 2*x_coarse + 2*hat{mu}.

    U_blocked = U_fine(2x) @ U_fine(2x + hat{mu})

    This is a 'decimation' type blocking (keeping the path of length 2).
    We then project back to SU(2) to ensure numerical stability.
    """
    L = U.shape[0]
    if L % 2 != 0:
        raise ValueError("Lattice size must be even for scale-2 blocking")

    L_blocked = L // 2

    # Initialize blocked lattice
    # We will compute each direction separately

    # Slice indices for stride 2
    # We want x in 0..L-1 with stride 2

    # Direction 0 (x)
    # U1 at x=0,2,4...
    U1_0 = U[0::2, 0::2, 0::2, 0::2, 0]
    # U2 at x=1,3,5... (shifted by 1 in x)
    U2_0 = U[1::2, 0::2, 0::2, 0::2, 0]
    U_blocked_0 = U1_0 @ U2_0

    # Direction 1 (y)
    U1_1 = U[0::2, 0::2, 0::2, 0::2, 1]
    # U2 shifted by 1 in y
    U2_1 = U[0::2, 1::2, 0::2, 0::2, 1]
    U_blocked_1 = U1_1 @ U2_1

    # Direction 2 (z)
    U1_2 = U[0::2, 0::2, 0::2, 0::2, 2]
    # U2 shifted by 1 in z
    U2_2 = U[0::2, 0::2, 1::2, 0::2, 2]
    U_blocked_2 = U1_2 @ U2_2

    # Direction 3 (t)
    U1_3 = U[0::2, 0::2, 0::2, 0::2, 3]
    # U2 shifted by 1 in t
    U2_3 = U[0::2, 0::2, 0::2, 1::2, 3]
    U_blocked_3 = U1_3 @ U2_3

    # Stack results
    # Shape: (L/2, L/2, L/2, L/2, 4, 2, 2)
    U_blocked = jnp.stack([U_blocked_0, U_blocked_1, U_blocked_2, U_blocked_3], axis=4)

    # Project back to SU(2) to clean up any numerical drift
    # (Though product of SU(2) matrices is SU(2), float errors can accumulate)
    # We map to algebra and back

    # Flatten to apply vmap over all links
    flat_shape = (-1, 2, 2)
    U_flat = U_blocked.reshape(flat_shape)

    # Vectorized projection
    alphas = vmap(log_map_su2)(U_flat)
    U_projected_flat = vmap(exp_map_pauli)(alphas)

    return U_projected_flat.reshape(U_blocked.shape)

# --- OBSERVABLES ---

@jit
def measure_plaquette_defects(U, r_crit):
    """Measure fraction of plaquettes violating convexity bound"""
    Vol = U.shape[0]**4
    defects = 0.0
    total_plaqs = 0.0

    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))

            tr_vals = jnp.real(jnp.trace(P, axis1=-2, axis2=-1))
            tr_vals = jnp.clip(tr_vals, -2.0, 2.0)
            thetas = jnp.arccos(0.5 * tr_vals)

            defects += jnp.sum(thetas > r_crit)
            total_plaqs += Vol

    return defects / total_plaqs

@jit
def measure_plaquette_action_statistics(U, beta):
    """Measure mean and std of plaquette action distribution"""
    Vol = U.shape[0]**4
    actions = []

    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
            tr_vals = jnp.real(jnp.trace(P, axis1=-2, axis2=-1))

            # S_plaq = -beta/2 * Re Tr P
            S_plaq = -0.5 * beta * tr_vals
            actions.append(S_plaq.flatten())

    all_actions = jnp.concatenate(actions)
    return jnp.mean(all_actions), jnp.std(all_actions)

@jit
def estimate_local_convexity_radius(U, beta, n_samples=100):
    """
    Estimate effective local convexity radius by sampling small perturbations
    around typical configurations and checking Hessian eigenvalues.

    Returns: estimate of the radius where action is locally convex
    """
    # Sample random links
    L = U.shape[0]
    key = random.PRNGKey(42)

    # Take a few random links and perturb them slightly
    # Compute local curvature (second derivative of action)
    # This is a simplified proxy - full implementation would compute Hessian

    # For now, return plaquette-based estimate
    # Effective radius ~ 1/sqrt(local curvature)

    # Compute typical plaquette trace
    avg_plaq_trace = 0.0
    count = 0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_mu_nu = jnp.roll(U_mu, -1, axis=nu)
            U_nu_mu = jnp.roll(U_nu, -1, axis=mu)

            P = U_mu @ U_nu_mu @ jnp.conjugate(jnp.swapaxes(U_mu_nu, -1, -2)) @ jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
            avg_plaq_trace += jnp.mean(jnp.real(jnp.trace(P, axis1=-2, axis2=-1)))
            count += 1

    avg_plaq_trace /= count

    # Estimate curvature: higher plaq trace -> tighter binding -> larger effective radius
    # r_eff ~ arccos(trace/2) measures typical "distance from identity"
    r_eff = jnp.arccos(jnp.clip(0.5 * avg_plaq_trace, -1.0, 1.0))

    return r_eff

# --- MAIN RG STUDY DRIVER ---

def run_rg_blocking_study(L, beta, n_sweeps=2000, block_levels=2):
    """
    Run RG blocking study for a single (L, beta) point.

    Args:
        L: Lattice size (must be divisible by 2^block_levels)
        beta: Coupling parameter
        n_sweeps: Number of measurement sweeps
        block_levels: Number of RG blocking levels to perform
    """
    print("\n" + "="*70)
    print(f"RG BLOCKING STUDY: L={L}^4, Beta={beta:.4f}")
    print("="*70)

    r_crit = 1.9248  # Balaban's convexity threshold

    # Check lattice size
    if L % (2**block_levels) != 0:
        raise ValueError(f"L={L} must be divisible by 2^{block_levels}={2**block_levels}")

    # Initialize
    key = random.PRNGKey(42)
    U = init_lattice_cold(L)
    parity_mask = make_checkerboard_mask(L)

    @jit
    def full_sweep(key, U):
        key, U, acc1 = update_checkerboard(key, U, beta, parity_mask)
        key, U, acc2 = update_checkerboard(key, U, beta, 1 - parity_mask)
        return key, U, (acc1 + acc2) / 2.0

    # Thermalization
    print("\nThermalizing...")
    therm_sweeps = min(500, n_sweeps // 2)
    t0 = time.time()
    key, U, acc = full_sweep(key, U)
    acc.block_until_ready()
    print(f"JIT compile: {time.time()-t0:.2f}s")

    for i in range(therm_sweeps):
        key, U, acc = full_sweep(key, U)
        if i % 100 == 0:
            acc.block_until_ready()
            print(f"  Thermalization sweep {i}/{therm_sweeps}, acc={float(acc):.2f}")

    print(f"Thermalization complete: {time.time()-t0:.2f}s")

    # Measurements
    print(f"\nMeasuring unblocked lattice ({n_sweeps} sweeps)...")
    t0 = time.time()

    measurements_unblocked = []

    for i in range(n_sweeps):
        key, U, acc = full_sweep(key, U)

        if i % 50 == 0:
            # Measure observables
            rho_defect = measure_plaquette_defects(U, r_crit)
            r_eff = estimate_local_convexity_radius(U, beta)
            S_mean, S_std = measure_plaquette_action_statistics(U, beta)

            measurements_unblocked.append({
                "sweep": i,
                "defect_density": float(rho_defect),
                "convexity_radius": float(r_eff),
                "plaq_action_mean": float(S_mean),
                "plaq_action_std": float(S_std),
                "acceptance": float(acc)
            })

            if i % 200 == 0 and i > 0:
                print(f"  Sweep {i}: defect={float(rho_defect):.6f}, r_eff={float(r_eff):.4f}, acc={float(acc):.2f}")

    print(f"Unblocked measurements: {time.time()-t0:.2f}s")

    # Perform RG blocking and measure at each level
    blocked_results = []
    U_current = U

    for level in range(1, block_levels + 1):
        print(f"\n--- RG Blocking Level {level} ---")
        print(f"Blocking from L={U_current.shape[0]} to L={U_current.shape[0]//2}...")

        t0 = time.time()
        U_blocked = perform_rg_blocking(U_current)
        print(f"Blocking transformation: {time.time()-t0:.2f}s")

        # Measure blocked lattice
        print("Measuring blocked lattice...")
        rho_defect_blocked = measure_plaquette_defects(U_blocked, r_crit)
        r_eff_blocked = estimate_local_convexity_radius(U_blocked, beta)
        S_mean_blocked, S_std_blocked = measure_plaquette_action_statistics(U_blocked, beta)

        result = {
            "level": level,
            "L_blocked": U_blocked.shape[0],
            "defect_density": float(rho_defect_blocked),
            "convexity_radius": float(r_eff_blocked),
            "plaq_action_mean": float(S_mean_blocked),
            "plaq_action_std": float(S_std_blocked)
        }

        blocked_results.append(result)

        print(f"  L={result['L_blocked']}: defect={result['defect_density']:.6f}, r_eff={result['convexity_radius']:.4f}")

        U_current = U_blocked

    # Compute statistics
    skip = len(measurements_unblocked) // 10
    analysis_data = measurements_unblocked[skip:]

    summary = {
        "L": L,
        "beta": beta,
        "n_sweeps": n_sweeps,
        "block_levels": block_levels,
        "r_crit": r_crit,
        "unblocked": {
            "mean_defect_density": float(np.mean([m["defect_density"] for m in analysis_data])),
            "std_defect_density": float(np.std([m["defect_density"] for m in analysis_data])),
            "mean_convexity_radius": float(np.mean([m["convexity_radius"] for m in analysis_data])),
            "mean_plaq_action": float(np.mean([m["plaq_action_mean"] for m in analysis_data])),
            "std_plaq_action": float(np.mean([m["plaq_action_std"] for m in analysis_data])),
        },
        "blocked_levels": blocked_results,
        "measurements_unblocked": measurements_unblocked
    }

    # Estimate κ_eff
    print("\n" + "="*70)
    print("EFFECTIVE POLYMER ACTIVITY ESTIMATE")
    print("="*70)

    for i, result in enumerate(blocked_results):
        level = result["level"]
        rho = result["defect_density"]

        # κ_eff ≲ ρ_defect × exp(-ΔS_local)
        # Typical defect action excess ~ beta (rough estimate)
        Delta_S = beta * 0.5  # Conservative estimate
        kappa_eff = rho * np.exp(-Delta_S)

        print(f"Level {level}: ρ={rho:.2e}, κ_eff ≲ {kappa_eff:.2e}")
        result["kappa_eff_estimate"] = float(kappa_eff)

    print("\nBalaban's proof requires κ to remain small at ALL scales.")
    print("Validation: κ_eff should DECREASE under blocking.")
    print("="*70)

    return summary

# --- MULTI-PARAMETER SCAN ---

def run_full_rg_study():
    """
    Run RG blocking study for multiple parameters as recommended by GPT-5.1.

    Parameters from guidance:
    - L: [8, 12, 16]
    - Beta: 3 values from 4.6 to 5.4
    - n_sweeps: 2000
    """

    print("="*70)
    print("COMPLETE RG BLOCKING STUDY")
    print("Based on GPT-5.1 Consultation Guidance")
    print("="*70)

    # Parameters
    lattice_sizes = [8, 12, 16]
    beta_values = [4.6, 5.0, 5.4]
    n_sweeps = 2000

    all_results = []

    for L in lattice_sizes:
        # Determine max blocking levels
        max_levels = int(np.log2(L)) - 1  # Keep at least L=4 after blocking
        max_levels = min(max_levels, 2)  # Limit to 2 levels for reasonable compute

        for beta in beta_values:
            print(f"\n{'#'*70}")
            print(f"# CONFIGURATION: L={L}, Beta={beta:.2f}")
            print(f"{'#'*70}")

            result = run_rg_blocking_study(
                L=L,
                beta=beta,
                n_sweeps=n_sweeps,
                block_levels=max_levels
            )

            all_results.append(result)

    # Save complete results
    output = {
        "study_type": "rg_blocking_multi_parameter",
        "timestamp": datetime.now().isoformat(),
        "gpt51_guidance": "next_simulation_guidance.json",
        "results": all_results
    }

    print("\n" + "="*70)
    print("COMPLETE RESULTS JSON")
    print("="*70)
    print(json.dumps(output, indent=2))

    return output

# --- MAIN EXECUTION ---

if __name__ == "__main__":
    # Option 1: Run single configuration for testing
    # result = run_rg_blocking_study(L=8, beta=5.0, n_sweeps=500, block_levels=2)

    # Option 2: Run full multi-parameter study (recommended)
    results = run_full_rg_study()

    print("\n✅ RG Blocking Study Complete!")
    print("Copy the JSON output to analyze κ_eff evolution under blocking.")


JAX Device: cuda:0
JAX Backend: gpu
COMPLETE RG BLOCKING STUDY
Based on GPT-5.1 Consultation Guidance

######################################################################
# CONFIGURATION: L=8, Beta=4.60
######################################################################

RG BLOCKING STUDY: L=8^4, Beta=4.6000

Thermalizing...
JIT compile: 5.43s
  Thermalization sweep 0/500, acc=0.70
  Thermalization sweep 100/500, acc=0.70
  Thermalization sweep 200/500, acc=0.70
  Thermalization sweep 300/500, acc=0.71
  Thermalization sweep 400/500, acc=0.70
Thermalization complete: 9.28s

Measuring unblocked lattice (2000 sweeps)...
  Sweep 200: defect=0.000041, r_eff=0.5978, acc=0.71
  Sweep 400: defect=0.000000, r_eff=0.5961, acc=0.70
  Sweep 600: defect=0.000000, r_eff=0.5954, acc=0.70
  Sweep 800: defect=0.000000, r_eff=0.6018, acc=0.71
  Sweep 1000: defect=0.000041, r_eff=0.5969, acc=0.70
  Sweep 1200: defect=0.000000, r_eff=0.5979, acc=0.71
  Sweep 1400: defect=0.000000, r_eff=0.5970, acc

In [ ]:
"""
q-Racah 6j-Symbol Calculator (JAX/GPU)

This script computes the q-deformed 6j-symbols for SU_q(2), which are the
fundamental building blocks of the Turaev-Viro/Yang-Mills tensor network.

It implements the Racah sum formula using JAX for massive parallelization
over the deformation angle theta (where q = e^{i theta}).

Physics Context:
- These 6j-symbols define the F-matrices of the fusion category.
- They encode the topological theta-term in the 4D YM lattice model.
- We need to understand their oscillatory behavior to control the sign problem.

Reference: Bridge Memo, Section 2.2
"""
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from functools import partial

# Enable 64-bit precision for factorials
jax.config.update("jax_enable_x64", True)

@jax.jit
def q_number(n, theta):
    """Computes [n]_q = (q^n - q^-n) / (q - q^-1) with q = exp(i*theta).
    This simplifies to sin(n*theta) / sin(theta).
    """
    # Use the limit value for small theta to avoid division by zero
    # [n]_q = sin(n*theta)/sin(theta)
    # For numerical stability, we use sinc-like formulation or just direct sin

    # Avoid division by zero at theta=0
    denom = jnp.sin(theta)
    safe_denom = jnp.where(jnp.abs(denom) < 1e-10, 1.0, denom)
    val = jnp.sin(n * theta) / safe_denom

    # Correct limit at theta=0 is n
    return jnp.where(jnp.abs(theta) < 1e-10, n, val)

@jax.jit
def log_q_factorial(n, theta):
    """Computes log([n]_q!) = sum_{k=1}^n log([k]_q)."""
    # We need a loop or a vectorized sum. Since n is integer scalar usually,
    # but here we might want to scan theta.
    # Let's assume n is a static integer for the 6j symbol (spins are fixed integers).
    # We sum log(sin(k*theta)/sin(theta)).

    # Note: [k]_q can be negative for large theta, so we track log magnitude and sign?
    # For unitary representations (q on unit circle), [n]_q is real but can be negative.
    # We will work with complex logs to handle the sign.

    k_vals = jnp.arange(1, n + 1)
    q_nums = q_number(k_vals, theta)
    return jnp.sum(jnp.log(q_nums + 0j))

@jax.jit
def q_triangle(a, b, c, theta):
    """Computes the log of the q-triangle coefficient."""
    # Delta_q(a,b,c) = sqrt( [a+b-c]! [a-b+c]! [-a+b+c]! / [a+b+c+1]! )
    # Arguments a,b,c are integers (2*spin).

    # Check triangle inequalities
    valid = (a+b >= c) & (b+c >= a) & (c+a >= b) & ((a+b+c)%2 == 0)

    term1 = log_q_factorial((a + b - c)//2, theta)
    term2 = log_q_factorial((a - b + c)//2, theta)
    term3 = log_q_factorial((-a + b + c)//2, theta)
    term4 = log_q_factorial((a + b + c)//2 + 1, theta)

    log_val = 0.5 * (term1 + term2 + term3 - term4)
    return jnp.where(valid, log_val, -jnp.inf) # Return -inf log for invalid

@jax.jit
def q_6j_symbol(j1, j2, j3, j4, j5, j6, theta):
    """
    Computes the q-6j symbol {j1 j2 j3}
                             {j4 j5 j6}_q
    All j inputs are DOUBLED spins (integers).
    """
    # 1. Triangle Coefficients
    # Triangles: (j1 j2 j3), (j1 j5 j6), (j4 j2 j6), (j4 j5 j3)
    # Note: Standard 6j definition triangles might differ slightly in ordering,
    # checking standard Racah form.
    # Standard triangles for {j1 j2 j3 / j4 j5 j6}:
    # (j1 j2 j3), (j1 j5 j6), (j4 j2 j6), (j4 j5 j3)

    t1 = q_triangle(j1, j2, j3, theta)
    t2 = q_triangle(j1, j5, j6, theta)
    t3 = q_triangle(j4, j2, j6, theta)
    t4 = q_triangle(j4, j5, j3, theta)

    log_prefactor = t1 + t2 + t3 + t4

    # 2. Summation
    # Range of k: max(T1, T2, T3, T4) <= k <= min(S1, S2, S3)
    # T1 = j1+j2+j3, etc... wait, the Racah formula is:
    # sum_z (-1)^z [z+1]! / ( [z-j1-j2-j3]! [z-j1-j5-j6]! ... )
    # Let's use the standard Racah formula adapted for q.

    # Variables for the sum (using doubled spins)
    # a=j1, b=j2, c=j3, d=j4, e=j5, f=j6
    a, b, c, d, e, f = j1, j2, j3, j4, j5, j6

    # Limits for z (summation index k in some texts, z here)
    # Lower bound: max(a+b+c, a+e+f, d+b+f, d+e+c) // 2
    # Upper bound: min(a+b+d+e, a+c+d+f, b+c+e+f) // 2

    k_min = jnp.max(jnp.array([a+b+c, a+e+f, d+b+f, d+e+c])) // 2
    k_max = jnp.min(jnp.array([a+b+d+e, a+c+d+f, b+c+e+f])) // 2

    # We create a mask for valid k
    # Since we can't have dynamic loop bounds easily in JAX jit, we scan a fixed range
    # and mask. Max possible spin sum is sum of all j.
    max_k_possible = (a+b+c+d+e+f) # Loose upper bound
    k_range = jnp.arange(max_k_possible + 1)

    mask = (k_range >= k_min) & (k_range <= k_max)

    def compute_term(k):
        # Denominators
        d1 = log_q_factorial(k - (a+b+c)//2, theta)
        d2 = log_q_factorial(k - (a+e+f)//2, theta)
        d3 = log_q_factorial(k - (d+b+f)//2, theta)
        d4 = log_q_factorial(k - (d+e+c)//2, theta)
        d5 = log_q_factorial((a+b+d+e)//2 - k, theta)
        d6 = log_q_factorial((a+c+d+f)//2 - k, theta)
        d7 = log_q_factorial((b+c+e+f)//2 - k, theta)

        num = log_q_factorial(k + 1, theta)

        log_val = num - (d1 + d2 + d3 + d4 + d5 + d6 + d7)

        # Phase (-1)^k
        phase = (-1.0)**k

        return phase * jnp.exp(log_val)

    # Vectorize over k_range
    terms = jax.vmap(compute_term)(k_range)
    sum_val = jnp.sum(jnp.where(mask, terms, 0.0))

    return jnp.exp(log_prefactor) * sum_val

def run_experiment():
    print("Running q-Racah 6j-Symbol Scan...")

    # Define a range of theta (0 to 2pi)
    thetas = jnp.linspace(0.01, 2*jnp.pi, 1000)

    # Configuration: Tetrahedron with all spins j=1 (doubled j=2)
    # {1 1 1}
    # {1 1 1}
    j_val = 2 # Spin 1

    # Vectorize the 6j function over theta
    v_q6j = jax.vmap(lambda t: q_6j_symbol(j_val, j_val, j_val, j_val, j_val, j_val, t))

    print("Computing...")
    results = v_q6j(thetas)

    # Convert to numpy for plotting
    thetas_np = np.array(thetas)
    results_np = np.array(results)

    print("Plotting...")
    plt.figure(figsize=(10, 6))
    plt.plot(thetas_np, results_np.real, label="Real Part")
    plt.plot(thetas_np, results_np.imag, label="Imag Part", linestyle="--")
    plt.title(f"q-Deformed 6j-Symbol vs Theta (Spins j=1)")
    plt.xlabel("Theta (radians)")
    plt.ylabel("{6j}_q")
    plt.legend()
    plt.grid(True)

    output_file = "q_racah_6j_scan.png"
    plt.savefig(output_file)
    print(f"Saved plot to {output_file}")

    # Save data
    np.savetxt("q_racah_data.txt", np.column_stack((thetas_np, results_np.real, results_np.imag)), header="Theta Real Imag")

if __name__ == "__main__":
    run_experiment()
